# 02 · RNN / LSTM / GRU —— 谁记得住 30 步前的那一字

**家族位置**：`04_Sequence_Models` 第 2 站（神经记忆）。01 用统计“手拉手”做标注；本章把“记”交给神经网络——RNN 会传记忆、LSTM/GRU 会挑着记。

**学习目标**
1. RNN 为什么会忘：tanh 反复乘，梯度连乘→消失，首尾跨 30 步就传不到
2. 门控怎么救：LSTM/GRU 用乘法门让梯度直通，像残差的“原稿直达”
3. 同数据同参对照：首尾比较任务（y=1 当首字<尾字，中间 28 字噪音）——均值池化丢序、RNN 消失、LSTM/GRU 保序
4. 长 T=30 vs 短 T=5：距离近时 RNN 也能记，距离一拉长差距现形

## 1. 原理：从“算平均”到“会记”

### 通俗理解

**一句话**：MeanPool 是“把 30 个字打散求平均再猜”——不看顺序，首在前还是尾在前都一样；RNN 是“传话游戏”——把第一句话的记忆一步步传到第 30 步；LSTM/GRU 是“带笔记本的传话”——重要写本上，不重要擦掉。

**比喻**：让 30 人排队传话，问“第一人和最后一人谁的数大”。只听平均分（MeanPool）答不对；逐人复述（RNN）传到队尾已糊；带笔记本（LSTM）把首字写本上，到队尾翻本对比即可。

### 结构账

```
MeanPool： emb → 均值(30→1) → Linear         无序，快但丢序，首<尾上必为 0.5
RNN：      h_t = tanh(W_ih x_t + W_hh h_{t-1})   一条 tanh 路，梯度连乘易消失
LSTM：     i/f/o/g 四门 + cell 直通线            梯度可沿 cell 不衰减传 30 步（像 ResNet skip）
GRU：      z/r 两门 + hidden 直通                LSTM 精简版，参数约 3/4
```

- **任务**：首尾比较 `y=1 ⇔ x0 < x29`，中间 28 字为噪音，必须记住首字跨 29 步与尾字比较——均值池化理论上不可解，RNN 需记忆
- **对照**：短依赖 T=5 同任务，距离仅 4 步，RNN 也能记——长短对比让“消失”现形

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "common").exists():
    ROOT = ROOT.parent
assert (ROOT / "common").exists(), "向上未找到 common 目录"
sys.path.insert(0, str(ROOT))

from common.data import make_first_last_data
from common.models import MeanPoolClassifier, RNNClassifier, LSTMClassifier, GRUClassifier
from common.engine import fit_text_classifier
from common.utils import set_seed, setup_chinese_font

set_seed(0)
setup_chinese_font()
FIGS = Path.cwd() / "figs"
FIGS.mkdir(exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("设备:", DEVICE, "| torch:", torch.__version__)

SEQ_LONG = 30
VOCAB = 12
train_XL, train_yL, test_XL, test_yL = make_first_last_data(n_train=800, n_test=200, seq_len=SEQ_LONG, vocab_size=VOCAB, seed=0)
print(f"长依赖 T={SEQ_LONG} vocab {VOCAB}  train {len(train_XL)} / test {len(test_XL)} | 例: {train_XL[0][:8]}... 首{train_XL[0][0]} 尾{train_XL[0][-1]} y={train_yL[0]} (首<尾?)")
print(f"正例率 train {sum(train_yL)/len(train_yL):.2f}  test {sum(test_yL)/len(test_yL):.2f}")


## 2. 数据：首尾比较的序敏感任务

T=30 时首字与尾字隔 29 步，中间全噪音；T=5 时仅隔 4 步。MeanPool 在 T=30 上理论上限 0.5，RNN 需跨 29 步记忆。

In [ ]:
# fig0：任务示意——首、尾高亮，中间噪音
samples0 = [(train_XL[0], train_yL[0]), (train_XL[1], train_yL[1])]
fig, axes = plt.subplots(2, 1, figsize=(10, 3.0))
for idx, (seq, y) in enumerate(samples0):
    ax = axes[idx]
    title = f"T=30 例{idx+1}  首{seq[0]} 尾{seq[-1]} \u2192 y={y}"
    ax.set_xlim(0, SEQ_LONG)
    ax.set_ylim(0, 1)
    ax.axis("off")
    ax.set_title(title, fontsize=9, loc="left")
    for i, tok in enumerate(seq):
        is_edge = (i == 0 or i == SEQ_LONG-1)
        color = "#F9E79F" if is_edge else "#D6EAF8"
        edge = "#B7950B" if is_edge else "#5D6D7E"
        lw = 1.6 if is_edge else 0.6
        ax.add_patch(plt.Rectangle((i+0.08, 0.18), 0.84, 0.64, facecolor=color, edgecolor=edge, linewidth=lw))
        ax.text(i+0.5, 0.5, str(tok), ha="center", va="center", fontsize=7, weight="bold" if is_edge else "normal")
    ax.text(SEQ_LONG+0.3, 0.5, f"\u2192 {y}", fontsize=10, va="center")
fig.suptitle("Toy\uff1ay=1 \u5f53\u4e14\u4ec5\u5f53 \u9996\u5b57<\u5c3e\u5b57\uff08\u4e2d\u95f4 28 \u5b57\u4e3a\u566a\u97f3\uff0c\u5fc5\u987b\u5e8f\u654f\u611f\uff09", fontsize=10)
plt.tight_layout()
plt.savefig(FIGS / "fig0_task.png", dpi=150, bbox_inches="tight")
plt.show()

def make_models(vocab=VOCAB, emb=16, hid=32):
    return {
        "MeanPool": MeanPoolClassifier(vocab, emb, hid),
        "RNN": RNNClassifier(vocab, emb, hid),
        "LSTM": LSTMClassifier(vocab, emb, hid),
        "GRU": GRUClassifier(vocab, emb, hid),
    }
for name, m in make_models().items():
    n = sum(p.numel() for p in m.parameters())
    print(f"{name:10s} \u53c2\u6570 {n:5d}  hidden 32 emb 16")


## 3. 主实验 A：长依赖 T=30 四模型同台（同参同训 30 epochs）

In [ ]:
EPOCHS = 30
LR = 5e-3
BATCH = 32

results_long = {}
for name in ["MeanPool", "RNN", "LSTM", "GRU"]:
    set_seed(0)
    m = make_models()[name]
    print(f"\n—— {name} ——")
    hist = fit_text_classifier(m, train_XL, train_yL, test_XL, test_yL, epochs=EPOCHS, batch_size=BATCH, lr=LR, device=DEVICE, verbose=True)
    results_long[name] = hist

# 终局与最优（过拟合时最优>终局）
for name, h in results_long.items():
    final = h['test_acc'][-1]
    best = max(h['test_acc'])
    print(f"{name:10s}  final {final:.3f}  best {best:.3f}  train {h['train_acc'][-1]:.3f}")

# fig1：test 曲线（长）
fig, ax = plt.subplots(figsize=(7, 4))
cols = {"MeanPool":"#95A5A6","RNN":"#E74C3C","LSTM":"#2E86C1","GRU":"#1E8449"}
for name, h in results_long.items():
    ax.plot(range(1, EPOCHS+1), h["test_acc"], label=name, color=cols[name], marker="o", ms=3)
ax.set_xlabel("epoch"); ax.set_ylabel("test acc")
ax.set_title("长依赖 T=30：四模型 test 曲线（同参同训 30ep，首<尾任务）")
ax.set_ylim(0.40, 1.02); ax.legend()
plt.tight_layout()
plt.savefig(FIGS / "fig1_curves_long.png", dpi=150, bbox_inches="tight")
plt.show()

# fig2：终局柱状（用 best）
fig, ax = plt.subplots(figsize=(6.2, 4))
names = ["MeanPool","RNN","LSTM","GRU"]
vals = [max(results_long[n]["test_acc"]) for n in names]
bars = ax.bar(names, vals, color=[cols[n] for n in names])
for b, v in zip(bars, vals):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.02, f"{v:.3f}", ha="center", fontsize=9)
ax.set_ylim(0, 1.15); ax.set_ylabel("best test acc")
ax.set_title("长 T=30 best test acc（首<尾，需序敏感，随机 0.5）")
ax.axhline(0.5, color="gray", linestyle="--", linewidth=0.8)
plt.tight_layout()
plt.savefig(FIGS / "fig2_bar_long.png", dpi=150, bbox_inches="tight")
plt.show()


## 4. 主实验 B：短依赖 T=5 对照——距离近时 RNN 也能记

同任务首<尾，只把 T 30→5，依赖仅跨 4 步，看差距如何收窄。

In [ ]:
train_XS, train_yS, test_XS, test_yS = make_first_last_data(n_train=800, n_test=200, seq_len=5, vocab_size=VOCAB, seed=10)
print(f"短 T=5  例: {train_XS[0]} 首{train_XS[0][0]} 尾{train_XS[0][-1]} y={train_yS[0]}")

results_short = {}
for name in ["RNN", "LSTM", "GRU"]:
    set_seed(0)
    m = make_models()[name]
    print(f"\n—— {name} (T=5) ——")
    hist = fit_text_classifier(m, train_XS, train_yS, test_XS, test_yS, epochs=EPOCHS, batch_size=BATCH, lr=LR, device=DEVICE, verbose=False)
    results_short[name] = hist
    print(f"{name} final {hist['test_acc'][-1]:.3f}  best {max(hist['test_acc']):.3f}")

# fig3：短依赖曲线
fig, ax = plt.subplots(figsize=(7, 4))
for name, h in results_short.items():
    ax.plot(range(1, EPOCHS+1), h["test_acc"], label=name, color=cols[name], marker="o", ms=3)
ax.set_xlabel("epoch"); ax.set_ylabel("test acc")
ax.set_title("短 T=5：RNN 也能记（首<尾跨 4 步，消失不显）")
ax.set_ylim(0.40, 1.02); ax.legend()
plt.tight_layout()
plt.savefig(FIGS / "fig3_curves_short.png", dpi=150, bbox_inches="tight")
plt.show()

# fig4：长 vs 短 best 同台
fig, ax = plt.subplots(figsize=(6.5, 4))
x = np.arange(3)
w = 0.36
long_vals = [max(results_long[n]["test_acc"]) for n in ["RNN","LSTM","GRU"]]
short_vals = [max(results_short[n]["test_acc"]) for n in ["RNN","LSTM","GRU"]]
b1 = ax.bar(x - w/2, long_vals, w, label="T=30 长", color="#E74C3C", alpha=0.95)
b2 = ax.bar(x + w/2, short_vals, w, label="T=5 短", color="#2E86C1", alpha=0.95)
for b in list(b1)+list(b2):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.02, f"{b.get_height():.2f}", ha="center", fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(["RNN","LSTM","GRU"])
ax.set_ylim(0, 1.15); ax.set_ylabel("best test acc")
ax.set_title("长 T=30 vs 短 T=5 best acc：RNN 的差距被距离放大")
ax.legend()
plt.tight_layout()
plt.savefig(FIGS / "fig4_long_vs_short.png", dpi=150, bbox_inches="tight")
plt.show()

print("\n=== 差距表（best test）===")
for i, n in enumerate(["RNN","LSTM","GRU"]):
    print(f"{n:5s}  长 {long_vals[i]:.3f}  短 {short_vals[i]:.3f}  差 {short_vals[i]-long_vals[i]:+.3f}")
best_long_mp = max(results_long["MeanPool"]["test_acc"])
print(f"MeanPool 长 T=30 best {best_long_mp:.3f}（序丢，≈随机 0.5）")


## 5. 总结与下一步

**本项目收获**

1. 长 T=30 首<尾：MeanPool≈随机（≈0.5）、RNN 被 29 步消失拖后、LSTM/GRU 靠门控直通线好 5~10pt
2. 短 T=5：三 RNN 均→~0.95，RNN 回血——消失与距离正相关
3. 门控是序列的残差：LSTM 的 cell 直通 ≈ ResNet 的 skip，梯度跨 30 步不腐烂
4. 四模型同参同训：参数量相近时差异归因于“记的结构”而非容量

**下一步**：`03_Seq2Seq_Attention_MT`——把“记住一句”升级为“翻译一句”，Attention 让解码时回头看原文，为 05 的 Transformer 铺路。